# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**URL:** https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll list all record sets and their associated fields from the dataset. You should reference entities by their `@id`.

In [ ]:
# List all record sets and their field IDs
record_sets = list(dataset.record_sets)

print("Available Record Sets:")
for rs in record_sets:
    print(f"- Record Set: {rs['@id']} (name: {rs['name']})")
    if 'fields' in rs:
        print("  Fields:")
        for f in rs['fields']:
            print(f"    - {f['@id']} (name: {f['name']})")
    else:
        print("  No fields found.")
print()
# For demonstration, show preview of records for the first record set
if record_sets:
    demo_recordset_id = record_sets[0]['@id']
    print(f"Sample records for record set '{demo_recordset_id}':")
    for x in dataset.records(record_set=demo_recordset_id):
        print(x)
        break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We use the `@id` for each record set. If the dataset contains multiple record sets, you can extend this code to loop over each.

In [ ]:
# Extract data from each record set
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]
print("Record Set IDs:", record_set_ids)

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for record set '{record_set_id}': {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for record set '{record_set_id}'.")

## 4. Exploratory Data Analysis (EDA)
Apply common data-processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Use the field `@id`s found above. For this dataset, typical numeric fields might include GAD-7, PHQ-9, or PSQ scores. Grouping by demographic fields can provide aggregate insight.

In [ ]:
# Select the first available record set for EDA
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Assume numeric fields may include 'GAD-7', 'PHQ-9', or 'PSQ'
    # Find a numeric field by inspecting columns
    numeric_field_candidates = [c for c in df.columns if any(x in c.lower() for x in ['gad', 'phq', 'psq', 'score', 'numeric'])]
    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
    else:
        # fallback to first numeric column
        numeric_cols = df.select_dtypes(include='number').columns
        numeric_field = numeric_cols[0] if len(numeric_cols) else None

    if numeric_field:
        print(f"Analyzing numeric field: {numeric_field}")

        threshold = df[numeric_field].mean()   # Use mean as threshold example
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the field
        filtered_df[numeric_field + '_normalized'] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

        # Group by demographic field (e.g., 'gender', 'age', 'marital_status', etc.)
        group_field_candidates = [c for c in df.columns if any(x in c.lower() for x in ['gender', 'age', 'marital', 'education', 'village'])]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we'll plot a histogram of the selected numeric field and a bar chart for the grouped data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure EDA data is available
if 'filtered_df' in locals() and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Barplot for group means
    if 'grouped_df' in locals():
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset enables nuanced analysis of mental health indicators and demographic associations in Kilifi County, Kenya.
- Using `mlcroissant`, metadata and records can be loaded and processed programmatically with rich schema-based referencing (`@id`).
- Numeric fields such as GAD-7, PHQ-9, and PSQ scores allow for quantitative filtering and normalization, revealing trends across population groups.
- Further analyses can build on these insights for community health initiatives and developing public health strategies.
